In [40]:
#IMPORTS
from fileinput import filename
import pandas as pd
import hvplot.pandas
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from st_aggrid import GridOptionsBuilder, AgGrid, GridUpdateMode, DataReturnMode
import holoviews as hv
import numpy as np
import sys
import os
from pathlib import Path, WindowsPath, PureWindowsPath
sys.path.append(str(Path.cwd().parent))
from Intelligence import Utils, cloud_data_collector, DataFramer, Helper
import matplotlib, matplotlib_inline
#%matplotlib inline
from AIML.FinAIML_LinReg import FinAIML_LinReg
from AIML.FinAIML_LSTM import FinAIML_LSTM
from AIML.FinAIML_SVR import FinAIML_SVR
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [41]:
## Welcome to Company Predictive Analytics
def GetSelectedTicker():
        ticker = 'NVDA'
        return ticker
    
ticker = GetSelectedTicker()
tickerdetails = None
TotalProfitLoss = 0


In [42]:
abspath = os.path.abspath("")
#print(abspath)
tickerdatapath1 = Path(abspath).parent
#print(tickerdatapath1)
location = tickerdatapath1 / 'TickerData' # type: ignore
#print(location)
dfTicker:pd.DataFrame = Utils.GetLocalTickerData(ticker, location)
#print(f'Loading ticker data for ticker {ticker}')
#print(dfTicker)
if(dfTicker is None or dfTicker.empty):
    dfTicker = cloud_data_collector.GetYFinanceData(ticker, '5y')
    if(dfTicker is not None):
        dfTicker.drop('Adj Close', axis=1, inplace=True)
        Utils.SaveTickerData(ticker, dfTicker, location)

if(dfTicker is None or dfTicker.empty):
    print(f"Error loading data for ticker {ticker}.")             
else:
    print(f'Loading ticker data for ticker {ticker}')
    print(dfTicker.head())
    print(dfTicker.tail())


Loading ticker data for ticker NVDA
               Open     High      Low    Close      Volume
Date                                                      
2019-08-15  3.77075  3.79075  3.68475  3.71925   477200000
2019-08-16  3.98375  4.03850  3.92200  3.98900  1007960000
2019-08-19  4.10300  4.28550  4.08675  4.26950   815112000
2019-08-20  4.26400  4.26450  4.17550  4.19675   463188000
2019-08-21  4.26325  4.33625  4.24150  4.28075   427244000
                  Open        High         Low       Close     Volume
Date                                                                 
2024-08-08  102.000000  105.500000   97.519997  104.970001  391910000
2024-08-09  105.639999  106.599998  103.430000  104.750000  290844200
2024-08-12  106.320000  111.070000  106.260002  109.019997  325559900
2024-08-13  112.440002  116.230003  111.580002  116.139999  312646700
2024-08-14  118.529999  118.599998  114.070000  118.080002  338340200


In [43]:
dfPlot = dfTicker.hvplot.line(
    height=600,
    rot=90,
    responsive=True
)
hv.render(dfPlot, backend='bokeh')

Figure(id='1702', ...)

In [44]:
#Check for Null Values
dataHasNulls = dfTicker.isnull().values.any()
print(f'Null Value Present: {dataHasNulls}')
if(dataHasNulls):
    # Drop missing value
    dfTicker.fillna(value=0, inplace=True)
print(f'Null Value Present: {dfTicker.isnull().values.any()}')


Null Value Present: False
Null Value Present: False


In [45]:
#Get the Difference between today and previous day
dfTicker['Return'] = dfTicker.Close.diff(periods=1)
dfTicker['Return'].fillna(0, inplace=True)
# Calculate the daily returns using the closing prices and the pct_change function
dfTicker["PercentReturn"] = dfTicker["Close"].pct_change()
dfTicker['PercentReturn'].fillna(0, inplace=True)

print(dfTicker.head())
print(dfTicker.tail())


               Open     High      Low    Close      Volume   Return  PercentReturn
Date                                                                              
2019-08-15  3.77075  3.79075  3.68475  3.71925   477200000  0.00000       0.000000
2019-08-16  3.98375  4.03850  3.92200  3.98900  1007960000  0.26975       0.072528
2019-08-19  4.10300  4.28550  4.08675  4.26950   815112000  0.28050       0.070318
2019-08-20  4.26400  4.26450  4.17550  4.19675   463188000 -0.07275      -0.017039
2019-08-21  4.26325  4.33625  4.24150  4.28075   427244000  0.08400       0.020015
                  Open        High         Low       Close     Volume    Return  PercentReturn
Date                                                                                          
2024-08-08  102.000000  105.500000   97.519997  104.970001  391910000  6.059998       0.061268
2024-08-09  105.639999  106.599998  103.430000  104.750000  290844200 -0.220001      -0.002096
2024-08-12  106.320000  111.070000  106

In [46]:
dfTicker['DailyTrend'] = dfTicker['Return'].apply(Helper.CalcTrend)
dfTicker['DailyTrend'].fillna(0, inplace=True)
dfTicker.DailyTrend = dfTicker.DailyTrend.astype('int')
dfTicker['Trend'] = 0
dfTicker.Trend = dfTicker.Trend.astype('int')
print(dfTicker.head())
print(dfTicker.tail())


               Open     High      Low    Close      Volume   Return  PercentReturn  DailyTrend  Trend
Date                                                                                                 
2019-08-15  3.77075  3.79075  3.68475  3.71925   477200000  0.00000       0.000000           0      0
2019-08-16  3.98375  4.03850  3.92200  3.98900  1007960000  0.26975       0.072528           1      0
2019-08-19  4.10300  4.28550  4.08675  4.26950   815112000  0.28050       0.070318           1      0
2019-08-20  4.26400  4.26450  4.17550  4.19675   463188000 -0.07275      -0.017039          -1      0
2019-08-21  4.26325  4.33625  4.24150  4.28075   427244000  0.08400       0.020015           1      0
                  Open        High         Low       Close     Volume    Return  PercentReturn  DailyTrend  Trend
Date                                                                                                             
2024-08-08  102.000000  105.500000   97.519997  104.970001

In [47]:
# Using multiple apply arguments
prevdailytrend = 0
prevtrend = 0
dailytrend = 0
trend = 0

for i in range(len(dfTicker)):
   #print(dfTicker.loc[i, "Name"], dfTicker.loc[i, "Age"])
    #previousrow = dfTicker.iloc[[i-1]]
    dailytrend = dfTicker["DailyTrend"].iloc[i]
    if(i==0): #First Row of the DataSet
        if(dailytrend is np.nan or dailytrend==0):
            dfTicker.loc[dfTicker.index[i], "Trend"] = 0
    else:
        prevdailytrend = dfTicker["DailyTrend"].iloc[i-1]
        prevtrend = dfTicker["Trend"].iloc[i-1]
        trend = 0
        if(prevtrend>=0):                       # type:ignore
            if(dailytrend>=0):                  # type:ignore
                trend = prevtrend + dailytrend  # type:ignore
            elif( dailytrend<0):                # type:ignore
                trend = dailytrend
        elif(prevtrend<0):                      # type:ignore
            if(dailytrend>=0):                  # type:ignore
                trend = dailytrend
            elif( dailytrend<0):                # type:ignore
                trend = prevtrend + dailytrend # type:ignore
        dfTicker.loc[dfTicker.index[i], "Trend"] = trend
print(dfTicker.head())
print(dfTicker.tail())


               Open     High      Low    Close      Volume   Return  PercentReturn  DailyTrend  Trend
Date                                                                                                 
2019-08-15  3.77075  3.79075  3.68475  3.71925   477200000  0.00000       0.000000           0      0
2019-08-16  3.98375  4.03850  3.92200  3.98900  1007960000  0.26975       0.072528           1      1
2019-08-19  4.10300  4.28550  4.08675  4.26950   815112000  0.28050       0.070318           1      2
2019-08-20  4.26400  4.26450  4.17550  4.19675   463188000 -0.07275      -0.017039          -1     -1
2019-08-21  4.26325  4.33625  4.24150  4.28075   427244000  0.08400       0.020015           1      1
                  Open        High         Low       Close     Volume    Return  PercentReturn  DailyTrend  Trend
Date                                                                                                             
2024-08-08  102.000000  105.500000   97.519997  104.970001

In [48]:


# Set the variables for short window and long window periods
sma9 = 9
sma20 = 20
sma50 = 50
sma100 = 100
sma200 = 200
print('Generate the SMAs')
dfTicker["SMA9"] = dfTicker["Close"].rolling(window=sma9).mean()
dfTicker["SMA20"] = dfTicker["Close"].rolling(window=sma20).mean()
dfTicker["SMA50"] = dfTicker["Close"].rolling(window=sma50).mean()
dfTicker["SMA100"] = dfTicker["Close"].rolling(window=sma100).mean()
dfTicker["SMA200"] = dfTicker["Close"].rolling(window=sma200).mean()

dfTicker['SMA9'].fillna(0, inplace=True)
dfTicker['SMA20'].fillna(0, inplace=True)
dfTicker['SMA50'].fillna(0, inplace=True)
dfTicker['SMA100'].fillna(0, inplace=True)
dfTicker['SMA200'].fillna(0, inplace=True)
dfTicker.SMA9 = dfTicker.SMA9.astype('float')
dfTicker.SMA20 = dfTicker.SMA20.astype('float')
dfTicker.SMA50 = dfTicker.SMA50.astype('float')
dfTicker.SMA100 = dfTicker.SMA100.astype('float')
dfTicker.SMA200 = dfTicker.SMA200.astype('float')

print(dfTicker)
print(f'dfTicker dtypes are {dfTicker.dtypes}')

Generate the SMAs
                  Open        High         Low       Close      Volume    Return  PercentReturn  DailyTrend  Trend        SMA9       SMA20      SMA50     SMA100     SMA200
Date                                                                                                                                                                       
2019-08-15    3.770750    3.790750    3.684750    3.719250   477200000  0.000000       0.000000           0      0    0.000000    0.000000    0.00000    0.00000   0.000000
2019-08-16    3.983750    4.038500    3.922000    3.989000  1007960000  0.269750       0.072528           1      1    0.000000    0.000000    0.00000    0.00000   0.000000
2019-08-19    4.103000    4.285500    4.086750    4.269500   815112000  0.280500       0.070318           1      2    0.000000    0.000000    0.00000    0.00000   0.000000
2019-08-20    4.264000    4.264500    4.175500    4.196750   463188000 -0.072750      -0.017039          -1     -1    0.00

In [49]:
lstDecimalCols = ['Open', 'High', 'Low', 'Close', 'SMA9', 'SMA20', 'SMA50', 'SMA100', 'SMA200']
dfTicker = DataFramer.SetDataFrameColsTo2Decimals(dfData=dfTicker, lstcols=lstDecimalCols)

dfTicker['Open'].fillna(0, inplace=True)
dfTicker['High'].fillna(0, inplace=True)
dfTicker['Low'].fillna(0, inplace=True)
dfTicker['Close'].fillna(0, inplace=True)
dfTicker['SMA9'].fillna(0, inplace=True)
dfTicker['SMA20'].fillna(0, inplace=True)
dfTicker['SMA50'].fillna(0, inplace=True)
dfTicker['SMA100'].fillna(0, inplace=True)
dfTicker['SMA200'].fillna(0, inplace=True)
dfTicker.Open = dfTicker.Open.astype('float')
dfTicker.High = dfTicker.High.astype('float')
dfTicker.Low = dfTicker.Low.astype('float')
dfTicker.Close = dfTicker.Close.astype('float')
dfTicker.SMA9 = dfTicker.SMA9.astype('float')
dfTicker.SMA20 = dfTicker.SMA20.astype('float')
dfTicker.SMA50 = dfTicker.SMA50.astype('float')
dfTicker.SMA100 = dfTicker.SMA100.astype('float')
dfTicker.SMA200 = dfTicker.SMA200.astype('float')

dfTicker = dfTicker.round(2)
print(dfTicker)
#print(f'dfTicker dtypes are {dfTicker.dtypes}')

              Open    High     Low   Close      Volume  Return  PercentReturn  DailyTrend  Trend    SMA9   SMA20   SMA50  SMA100  SMA200
Date                                                                                                                                    
2019-08-15    3.77    3.79    3.68    3.72   477200000    0.00           0.00           0      0    0.00    0.00    0.00    0.00    0.00
2019-08-16    3.98    4.04    3.92    3.99  1007960000    0.27           0.07           1      1    0.00    0.00    0.00    0.00    0.00
2019-08-19    4.10    4.29    4.09    4.27   815112000    0.28           0.07           1      2    0.00    0.00    0.00    0.00    0.00
2019-08-20    4.26    4.26    4.18    4.20   463188000   -0.07          -0.02          -1     -1    0.00    0.00    0.00    0.00    0.00
2019-08-21    4.26    4.34    4.24    4.28   427244000    0.08           0.02           1      1    0.00    0.00    0.00    0.00    0.00
...            ...     ...     ...     ..

In [50]:
print(dfTicker.columns)
featureColumns = ['Open', 'High', 'Low', 'Close', 'Volume', 'PercentReturn', 'Trend', 'SMA9', 'SMA20', 'SMA50', 'SMA100', 'SMA200']
# Start LinReg Orchestration
linregOrchestrator = FinAIML_LinReg(dfData=dfTicker, featureColumns=featureColumns, targetColName='Close', predictionColName='Predicted', train_data_percent=0.5, validation_data_percent=0.3, test_data_percent=0.5, initCapital=0 )
linregOrchestrator.Orchestrate()


Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'PercentReturn', 'DailyTrend', 'Trend', 'SMA9', 'SMA20', 'SMA50', 'SMA100', 'SMA200'], dtype='object')
            Open  High   Low      Volume  Return  PercentReturn  DailyTrend  Trend  SMA9  SMA20  SMA50  SMA100  SMA200  Close  Predicted
Date                                                                                                                                    
2019-08-15  3.77  3.79  3.68   477200000    0.00           0.00           0      0   0.0    0.0    0.0     0.0     0.0    NaN        NaN
2019-08-16  3.98  4.04  3.92  1007960000    0.27           0.07           1      1   0.0    0.0    0.0     0.0     0.0   3.99   3.527301
2019-08-19  4.10  4.29  4.09   815112000    0.28           0.07           1      2   0.0    0.0    0.0     0.0     0.0   4.27   3.819698
2019-08-20  4.26  4.26  4.18   463188000   -0.07          -0.02          -1     -1   0.0    0.0    0.0     0.0     0.0    NaN        NaN
2019-08-21  4.26

In [51]:
# Start LinReg Orchestration
featureColumns = ['Close', 'Volume', 'PercentReturn', 'Trend', 'SMA9', 'SMA20', 'SMA50', 'SMA100', 'SMA200']
linregOrchestrator1 = FinAIML_LinReg(dfData=dfTicker, featureColumns=featureColumns, targetColName='Close', predictionColName='Predicted', train_data_percent=0.5, validation_data_percent=0.3, test_data_percent=0.5, initCapital=0 )
linregOrchestrator1.Orchestrate()

            Open  High   Low      Volume  Return  PercentReturn  DailyTrend  Trend  SMA9  SMA20  SMA50  SMA100  SMA200  Close  Predicted
Date                                                                                                                                    
2019-08-15  3.77  3.79  3.68   477200000    0.00           0.00           0      0   0.0    0.0    0.0     0.0     0.0   3.72   3.275100
2019-08-16  3.98  4.04  3.92  1007960000    0.27           0.07           1      1   0.0    0.0    0.0     0.0     0.0    NaN        NaN
2019-08-19  4.10  4.29  4.09   815112000    0.28           0.07           1      2   0.0    0.0    0.0     0.0     0.0   4.27   3.843650
2019-08-20  4.26  4.26  4.18   463188000   -0.07          -0.02          -1     -1   0.0    0.0    0.0     0.0     0.0   4.20   3.694156
2019-08-21  4.26  4.34  4.24   427244000    0.08           0.02           1      1   0.0    0.0    0.0     0.0     0.0    NaN        NaN
              Open    High     Low     Vo

In [52]:
# Start SVM Orchestration
svmOrchestrator = FinAIML_SVR(dfData=dfTicker, featureColumns=featureColumns, targetColName='Close', predictionColName='Predicted', train_data_percent=0.5, validation_data_percent=0.3, test_data_percent=0.5, initCapital=0 )
svmOrchestrator.Orchestrate()

            Open  High   Low      Volume  Return  PercentReturn  DailyTrend  Trend  SMA9  SMA20  SMA50  SMA100  SMA200  Close  Predicted
Date                                                                                                                                    
2019-08-15  3.77  3.79  3.68   477200000    0.00           0.00           0      0   0.0    0.0    0.0     0.0     0.0    NaN        NaN
2019-08-16  3.98  4.04  3.92  1007960000    0.27           0.07           1      1   0.0    0.0    0.0     0.0     0.0   3.99  17.080695
2019-08-19  4.10  4.29  4.09   815112000    0.28           0.07           1      2   0.0    0.0    0.0     0.0     0.0   4.27  17.551966
2019-08-20  4.26  4.26  4.18   463188000   -0.07          -0.02          -1     -1   0.0    0.0    0.0     0.0     0.0    NaN        NaN
2019-08-21  4.26  4.34  4.24   427244000    0.08           0.02           1      1   0.0    0.0    0.0     0.0     0.0   4.28  18.314992
              Open    High     Low     Vo